# AASIST - *Hugging Face*
## Evaluation & Benchmarking
Pre-trained model from [Hugging Face](https://huggingface.co/MTUCI/AASIST3)

### Objective
The primary objective of **AASIST3** is to provide a robust defense against speech deepfakes and synthetic audio attacks. It is designed specifically for the **ASVspoof 2024 Challenge**, aiming to distinguish between "bonafide" (human) speech and "spoof" (AI-generated or replayed) audio across diverse languages and recording conditions.

### Architecture Description
AASIST3 evolves the original AASIST (*Anti-spoofing with Adaptive Softmax and Instance-wise Temperature*) framework by integrating **Kolmogorov-Arnold Networks (KAN)** and Self-Supervised Learning (SSL) features.


The architecture is structured into the following functional stages:

1.  **SSL Feature Extraction:**
    The model utilizes a **Wav2Vec2** encoder as a front-end to extract high-level representations from raw audio waveforms. This allows the model to benefit from robust features learned during large-scale self-supervised pre-training.

2.  **KAN Bridge & Transformation:**
    Unlike traditional architectures that rely on standard MLP/Linear layers, AASIST3 incorporates **KAN Linear Layers**. These layers use learnable activation functions on the edges (splines) rather than fixed activations on nodes, allowing for more complex and efficient feature transformation.

3.  **Residual Encoding:**
    The extracted features pass through a series of **Residual Blocks** to capture hierarchical spectral and temporal patterns, ensuring stable gradient flow during training.

4.  **Graph Attention Networks (GAT):**
    To model the relationship between different segments of the audio signal, the architecture employs two specialized graph modules:
    * **GAT-S (Spatial):** Focuses on modeling dependencies across different frequency bins or feature dimensions.
    * **GAT-T (Temporal):** Focuses on modeling the long-term temporal dependencies of the speech signal.

5.  **Multi-branch Inference & Output:**
    The model uses four parallel inference branches integrated with **master tokens** to aggregate global information. The final classification (bonafide vs. spoof) is performed by an output layer also powered by KAN, which provides the final decision logic.

## Repo cloning and model import

In [1]:
!git clone https://github.com/mtuciru/AASIST3.git
!cd AASIST3

Cloning into 'AASIST3'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 69 (delta 22), reused 69 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 36.43 KiB | 4.55 MiB/s, done.
Resolving deltas: 100% (22/22), done.


### Dependencies installation

In [2]:
#!pip install -r '/content/AASIST3/requirements.txt'

In [3]:
# 1. Purge all potentially conflicting packages
!pip uninstall -y torch torchvision torchaudio torchcodec datasets

# 2. Install the strictly aligned PyTorch ecosystem (CUDA 12.1)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Install remaining dependencies, pinning datasets to the stable 2.x branch
!pip install transformers accelerate datasets==2.19.1 soundfile

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 76.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 48.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

> **RuntimeError:** Could not load libtorchcodec
This error is a binary linkage failure. It occurs when the Python wrapper for torchcodec cannot initialize its underlying C++ engine.

[Post adressing this issue](https://discuss.huggingface.co/t/issue-with-torchcodec-when-fine-tuning-whisper-asr-model/169315/2)

In [4]:
# Colab VM or Linux
#!apt-get update && apt-get install -y ffmpeg
#!pip install -U "datasets[audio]" "torch==2.8.*" "torchcodec==0.7.*"
# HF docs: audio decoding uses TorchCodec + FFmpeg
# https://huggingface.co/docs/datasets/en/audio_load

# Evaluation

In [5]:
import sys
import os

import torch
import torch.nn.functional as F
import torchaudio

import pandas as pd
import numpy as np

### Import pretrained model: *AASIST3*

In [6]:
# Add the repository root to the search path
repo_root = "/content/AASIST3/"
if repo_root not in sys.path:
    sys.path.append(repo_root)

In [7]:
# Disables the 'torchcodec' backend in torchaudio.
# Used to force the use of legacy backends (like ffmpeg or sox) or to avoid
# experimental decoder issues that might affect audio feature extraction consistency.
os.environ["TORCHAUDIO_USE_TORCHCODEC"] = "0"

In [8]:
from model import aasist3

# Load the model from Hugging Face Hub
model = aasist3.from_pretrained("MTUCI/AASIST3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

In [9]:
# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on device:", device)

Running on device: cuda


In [10]:
# Forces CUDA kernels to run synchronously.
# Essential for debugging; it ensures that GPU errors are reported at the
# exact line of Python code that triggered them, rather than asynchronously later.
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

### Dataset Preprocessing: *ASVspoof_2019_LA*

In [11]:
def preprocess_audio(audio_data, sr):
    # Convert numpy to torch tensor and ensure Float32
    audio = torch.from_numpy(audio_data).float().unsqueeze(0)

    # A. Resampling to 16kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        audio = resampler(audio)

    # B. Mono Check
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)

    # C. Pre-emphasis Filter: y[n] = x[n] - 0.97 * x[n-1]
    audio = torch.cat((audio[:, :1], audio[:, 1:] - 0.97 * audio[:, :-1]), dim=1)

    # D. Z-Score Normalization
    audio = (audio - audio.mean()) / (audio.std() + 1e-7)

    # E. Temporal Shaping (Exactly 64,600 samples)
    target_len = 64600
    current_len = audio.shape[1]

    if current_len < target_len:
        audio = F.pad(audio, (0, target_len - current_len))
    else:
        audio = audio[:, :target_len]

    return audio

### Old code, don't run

In [14]:
from datasets import load_dataset
from tqdm import tqdm

# 1. Setup Model
model.to(device)
model.eval()

# 2. Load Dataset (Streaming)
ds = load_dataset("Bisher/ASVspoof_2019_LA", split="test", streaming=True)

# 3. Collection Loop
scores = []
labels = []

print(f"Starting evaluation on {device}...")

for i, sample in enumerate(tqdm(ds, desc="Processing samples")):
    if max_samples and i >= max_samples:
        break

    # Extract data
    audio_data = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Ground Truth Label: 0 for Bonafide, 1 for Spoof
    is_spoof = sample["key"] == "spoof" # Bug detected
    label = 0 if is_spoof else 1

    # Preprocess
    processed_audio = preprocess_audio(audio_data, sr).to(device)

    # Inference
    with torch.no_grad():
        output = model(processed_audio)
        # We extract the probability for the "Bonafide" class (Index 0)
        # Standard EER/t-DCF scripts expect higher scores for human speech
        probs = torch.softmax(output, dim=1)
        bonafide_score = probs[0][0].item()

    scores.append(bonafide_score)
    labels.append(label)

print(f"\nCollection complete. Collected {len(scores)} samples.")

Starting evaluation on cuda...


Processing samples: 5000it [23:13,  3.59it/s]


Collection complete. Collected 5000 samples.


I am encountering the "Missing class" error because the **ASVspoof 2019 LA** dataset—specifically the test partition—is internally ordered. By processing only the first 5,000 examples out of roughly 71,000, I have restricted my subset to the **bonafide** (legitimate) section without ever reaching the **spoof** attacks.

In the original protocol, these samples are typically grouped alphabetically or by system. Given that the evaluation set contains 7,355 bonafide samples and 63,882 spoof samples, my first 5,000 items consist entirely of real audio.

**The Solution: Shuffling with a Buffer**
Since I am using `streaming=True`, I cannot shuffle the entire dataset instantaneously because it is not fully loaded into memory. I need to implement a **shuffle buffer** so that Hugging Face loads a specific number of samples and shuffles them before delivery.

```python
# Example of shuffling with a streaming dataset
shuffled_dataset = dataset.shuffle(seed=42, buffer_size=10_000)
```

By using a `buffer_size` larger than the initial block of bonafide samples, I ensure the model sees a mix of both classes even within a small subset.

### Fixed code

In [13]:
from datasets import load_dataset
from tqdm import tqdm

# 1. Setup Model
model.to(device)
model.eval()

# 2. Load Dataset (Streaming) con Shuffle
max_samples = 2000  # Set to None to evaluate the entire test set
seed = 42
buffer_size = 10000 # The higher, the better the shuffle, but demands more RAM
ds = load_dataset("Bisher/ASVspoof_2019_LA", split="test", streaming=True)
# We use .take() to avoid exhausting the whole streaming dataset
ds = ds.shuffle(seed=seed, buffer_size=buffer_size).take(max_samples)

In [14]:
from collections import Counter

# Inspect distribution of the selected samples to verify shuffling
print(f"Checking class distribution for {max_samples} samples...")

subset_labels = []
for sample in ds:
    # 1 is Bonafide, 0 is Spoof (mapping based on your earlier logic)
    label = "bonafide" if sample["key"] == 1 else "spoof"
    subset_labels.append(label)

distribution = Counter(subset_labels)
print(f"Class Distribution: {dict(distribution)}")

Checking class distribution for 2000 samples...
Class Distribution: {'bonafide': 1781, 'spoof': 219}


Now we've verified we actually have both bonafide and spoof classes, let's run evaluation.

In [26]:
# 3. Collection Loop
scores = []
labels = []

print(f"Starting evaluation on {device}...")

for i, sample in enumerate(tqdm(ds, desc="Processing samples")):
    if max_samples and i >= max_samples:
        break

    # Extract data
    audio_data = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Ground Truth Label: 1 for Bonafide, 0 for Spoof
    # Since sample["key"] already maps 1 to Bonafide and 0 to Spoof:
    label = sample["key"]

    # Preprocess
    processed_audio = preprocess_audio(audio_data, sr).to(device)

    # Inference
    with torch.no_grad():
        output = model(processed_audio)
        # We extract the probability for the "Bonafide" class (Index 0)
        # Standard EER/t-DCF scripts expect higher scores for human speech
        probs = torch.softmax(output, dim=1)
        bonafide_score = probs[0][0].item()

    scores.append(bonafide_score)
    labels.append(label)

print(f"\nCollection complete. Collected {len(scores)} samples.")

Starting evaluation on cuda...


Processing samples: 2000it [09:49,  3.40it/s]


Collection complete. Collected 2000 samples.


In [28]:
# Assuming scores and labels are populated lists
eval_results = pd.DataFrame({
    'score': scores,
    'label': labels
})

In [29]:
from utils.metrics import compute_eer

# Convert lists to numpy arrays to enable boolean indexing and .size attribute
np_scores = np.array(scores)
np_labels = np.array(labels)

bonafide_scores = np_scores[np_labels == 1]
spoof_scores = np_scores[np_labels == 0]

if bonafide_scores.size == 0 or spoof_scores.size == 0:
    print(f"Error: Missing class. Bonafide: {bonafide_scores.size}, Spoof: {spoof_scores.size}")
else:
    # compute_eer usually returns (eer, threshold) or (eer, frr, far, thresholds)
    # based on the AASIST3 repository structure
    results = compute_eer(bonafide_scores, spoof_scores)

    # Unpack based on the actual function return (handling potential length mismatches)
    if len(results) == 2:
        eer, threshold = results
    else:
        eer, frr, far, thresholds = results
        threshold = thresholds[np.nanargmin(np.abs(frr - far))]

    print(f"EER: {eer * 100:.4f}%")
    print(f"Optimal Threshold: {threshold}")

EER: 13.8117%
Optimal Threshold: 0.9513093829154968


In [53]:
for n in range(0, 100, 5):
    threshold = n / 100
    prediction = (eval_results['score'] >= threshold).astype(int)
    # General error flag
    is_error = eval_results['label'] != prediction
    print(f"Threshold: {threshold:.2f}, Errors: {is_error.sum()}")

Threshold: 0.00, Errors: 219
Threshold: 0.05, Errors: 171
Threshold: 0.10, Errors: 159
Threshold: 0.15, Errors: 160
Threshold: 0.20, Errors: 159
Threshold: 0.25, Errors: 157
Threshold: 0.30, Errors: 159
Threshold: 0.35, Errors: 159
Threshold: 0.40, Errors: 161
Threshold: 0.45, Errors: 162
Threshold: 0.50, Errors: 166
Threshold: 0.55, Errors: 163
Threshold: 0.60, Errors: 167
Threshold: 0.65, Errors: 170
Threshold: 0.70, Errors: 179
Threshold: 0.75, Errors: 185
Threshold: 0.80, Errors: 201
Threshold: 0.85, Errors: 211
Threshold: 0.90, Errors: 230
Threshold: 0.95, Errors: 277


In [54]:
threshold = 0.25

# Assuming 1 = Bonafide, 0 = Spoof
# A prediction is Bonafide if score >= threshold
eval_results['prediction'] = (eval_results['score'] >= threshold).astype(int)

# False Positive (FP): Spoof (0) predicted as Bonafide (1)
eval_results['is_fp'] = (eval_results['label'] == 0) & (eval_results['prediction'] == 1)

# False Negative (FN): Bonafide (1) predicted as Spoof (0)
eval_results['is_fn'] = (eval_results['label'] == 1) & (eval_results['prediction'] == 0)

# General error flag
eval_results['is_error'] = eval_results['label'] != eval_results['prediction']

In [56]:
eval_results.sum()

,0
score,1779.357067
label,1781.000000
prediction,1840.000000
is_fp,108.000000
is_fn,49.000000
is_error,157.000000


In [57]:
from utils.metrics import compute_tDCF

# t-DCF constants for ASVspoof 2019 LA (Standard values)
Pspoof = 0.05
Ptar = 0.9405
Pnon = 0.0095
Cmiss = 1
Cfa = 10

# Ensure we have numpy arrays from the evaluation loop
np_scores = np.array(scores)
np_labels = np.array(labels)

# Separate bonafide and spoof scores
# 1 = Bonafide, 0 = Spoof
bonafide_scores = np_scores[np_labels == 1]
spoof_scores = np_scores[np_labels == 0]

if bonafide_scores.size == 0 or spoof_scores.size == 0:
    print(f"Insufficient data for t-DCF. Bonafide: {bonafide_scores.size}, Spoof: {spoof_scores.size}")
else:
    # Note: compute_tDCF usually requires CM scores and ASV scores.
    # Since we are evaluating the CM in isolation, we provide the CM scores.
    # In many implementations, compute_tDCF might require a specific format or aligned ASV scores.
    # Here we call it with the parameters extracted from your current results.
    try:
        # tdcf_value = compute_tDCF(bonafide_scores, spoof_scores, Pspoof, Ptar, Pnon, Cmiss, Cfa)
        # Removed Pnon to match the 6-argument signature
        tdcf_value = compute_tDCF(bonafide_scores, spoof_scores, Pspoof, Ptar, Cmiss, Cfa)
        print(f"t-DCF: {tdcf_value:.4f}")
    except Exception as e:
        print(f"Could not compute t-DCF: {e}")
        print("Check if compute_tDCF in your utils expects (bonafide_scores, spoof_scores) or a different signature.")

Could not compute t-DCF: 'int' object is not subscriptable
Check if compute_tDCF in your utils expects (bonafide_scores, spoof_scores) or a different signature.


In [58]:
import inspect
from utils.metrics import compute_tDCF

print("Function signature:", inspect.signature(compute_tDCF))

Function signature: (bonafide_score_cm, spoof_score_cm, Pfa_asv, Pmiss_asv, Pmiss_spoof_asv, cost_model)


In [72]:
try:
    # Results are usually (tDCF_norm, CM_EER) or a numpy array of costs
    results = compute_tDCF(
        bonafide_scores,
        spoof_scores,
        Pfa_asv,
        Pmiss_asv,
        Pmiss_spoof_asv,
        cost_model
    )

    # Check if result is a tuple/list or a numpy array
    if isinstance(results, (tuple, list, np.ndarray)):
        # If it's an array, we take the minimum or the first element
        t_dcf = np.min(results[0]) if isinstance(results[0], np.ndarray) else results[0]

        # Try to print EER if it's the second element
        if len(results) > 1:
            eer_val = results[1]
            # Handle eer_val being an array too
            if isinstance(eer_val, np.ndarray): eer_val = eer_val.item() if eer_val.size == 1 else eer_val[0]
            print(f"CM EER (internal): {eer_val * 100:.4f}%")
    else:
        t_dcf = results

    # Final print with a safety check for NaNs/Arrays
    output_val = t_dcf.item() if hasattr(t_dcf, 'item') else t_dcf
    print(f"t-DCF: {output_val:.4f}")

except Exception as e:
    print(f"Calculation failed: {e}")
    print("Note: Divide by zero errors often mean Pfa_asv or Pmiss_asv are 0 or the score distribution is too narrow.")

CM EER (internal): 2.5317%
t-DCF: nan


---

### Dataset class

In [86]:
from torch.utils.data import Dataset, DataLoader

class ASVspoofDataset(Dataset):
    def __init__(self, hf_dataset, max_samples=None):
        # Converting streaming dataset to list for indexed access
        self.data = list(hf_dataset.take(max_samples)) if max_samples else list(hf_dataset)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        audio_data = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]

        # Preprocess to tensor [64600]
        feature = preprocess_audio(audio_data, sr).squeeze(0)

        utterance_id = item["audio_file_name"]

        # CRITICAL FIX: Ensure label is long and within [0, 1]
        # Mapping: 1 (Bonafide) -> 1, 0 (Spoof) -> 0
        # Our ModelWrapper swaps [B, S] to [S, B],
        # so Index 0 = Spoof, Index 1 = Bonafide.
        label = int(item["key"])

        return feature, utterance_id, torch.tensor(label, dtype=torch.long)

You should download the official evaluation trial file from [Kaggle](https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset/data)
> Look into: ´LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt´

[Documentation](https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset/data)

In [87]:
# Initialize Dataset and Loader
max_samples=500
eval_ds = ASVspoofDataset(ds, max_samples=max_samples)
eval_loader = DataLoader(eval_ds, batch_size=32, shuffle=False, num_workers=2)

# Placeholders for paths
save_path = "results/scores.txt"
trial_path = "protocols/ASVspoof2019.LA.cm.eval.trl.txt" # Path to official protocol
loss_fn = torch.nn.CrossEntropyLoss()

### Dataset wrapper

In [88]:
import torch.nn as nn

class ModelWrapper(nn.Module):
    def __init__(self, original_model):
        super(ModelWrapper, self).__init__()
        self.model = original_model

    def forward(self, x, **kwargs):
        # 1. Catch and ignore 'random' and 'dropout' via **kwargs
        out = self.model(x)

        # 2. Check if it's a tuple (features, logits) and unpack
        if isinstance(out, tuple):
            # In AASIST models, logits are typically the second element
            logits = out[1]
            features = out[0]
        else:
            logits = out
            features = torch.zeros((logits.size(0), 1), device=x.device)

        # 3. Handle Index Mismatch
        # The library's produce_evaluation_file uses batch_out[:, 1] for Bonafide.
        # As our model's Bonafide is at Index 0, we must swap them.
        # [Bonafide, Spoof] -> [Spoof, Bonafide]
        swapped_logits = logits[:, [1, 0]]

        return features, swapped_logits

In [89]:
# Wrap your existing model
model_wr = ModelWrapper(model)
model_wr.to(device)
model_wr.eval()
print("Wrapping completed")

Wrapping completed


## Evaluation file

In [90]:
!mkdir -p protocols
print("Directory 'protocols' created or already exists.")

Directory 'protocols' created or already exists.


In [92]:
from utils.metrics import produce_evaluation_file

# 1. Create a mini-protocol file from your 500 samples
mini_trial_path = "protocols/mini_eval_protocol.txt"

with open(mini_trial_path, "w") as f:
    # Iterate over the internal list of dictionaries in your dataset
    for item in eval_ds.data:
        utt_id = item["audio_file_name"]
        # ASVspoof protocol needs the string "bonafide" or "spoof"
        label_str = item["key"]

        # Format: [Speaker] [Utterance] [-] [SystemID] [Label]
        # We use dummy 'LA_0000' and '-' for compatibility
        f.write(f"LA_0000 {utt_id} - - {label_str}\n")

print(f"Mini protocol created with {len(eval_ds.data)} lines.")

Mini protocol created with 500 lines.


Load file manually before running next cell.

In [84]:
!ls 'protocols/'

ASVspoof2019.LA.cm.eval.trl.txt  mini_eval_protocol.txt


In [93]:
from utils.metrics import produce_evaluation_file

# Diagnosis: The CUDA assert usually triggers when cross_entropy receives a label
# outside [0, C-1]. Since we wrapped the model to swap indices [1, 0],
# we must ensure the 'label' passed in eval_loader matches this new alignment.

# Re-running with explicit trial_path pointing to our generated mini-protocol
# and ensuring the loss_fn is handled safely.
try:
    produce_evaluation_file(
        data_loader=eval_loader,
        model=model_wr,
        device=device,
        loss_fn=loss_fn,
        save_path=save_path,
        trial_path="protocols/mini_eval_protocol.txt",
        max_batches=None
    )
except RuntimeError as e:
    print(f"Caught expected CUDA error: {e}")
    print("Resetting GPU context is recommended (Restart Session) if errors persist.")

Caught expected CUDA error: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Resetting GPU context is recommended (Restart Session) if errors persist.


In [94]:
# Diagnostic: Check for labels outside the [0, 1] range in the loader
label_counts = {}
all_valid = True

print("Checking labels in eval_loader...")
for batch_idx, (_, _, labels) in enumerate(eval_loader):
    unique_labels = labels.unique().tolist()
    for l in unique_labels:
        label_counts[l] = label_counts.get(l, 0) + (labels == l).sum().item()
        if l not in [0, 1]:
            print(f"!!! INVALID LABEL FOUND: {l} at batch {batch_idx} !!!")
            all_valid = False

if all_valid:
    print("All labels are valid (0 or 1).")
else:
    print("Found invalid labels. These must be corrected to avoid CUDA asserts.")

print(f"Label distribution: {label_counts}")

Checking labels in eval_loader...
All labels are valid (0 or 1).
Label distribution: {0: 61, 1: 439}


In [ ]:
import os
# Set the environment variable to enable Device-Side Assertions
os.environ['TORCH_USE_CUDA_DSA'] = '1'

print('TORCH_USE_CUDA_DSA enabled. Please re-run your imports and model initialization.')

---

## Alternative strategy 1: Running metrics functions

In [ ]:
from utils.metrics import compute_eer, compute_det_curve, compute_tDCF

In [ ]:
%%writefile .env
# Path to the ASVspoof 2019/2021/2024 dataset
DATASET_DIR=/path/to/your/dataset

# Path to the pre-trained weights or the model you want to validate
CHECKPOINT_PATH=./models/weights.pth

# Optional: WandB configuration for logging validation metrics
WANDB_API_KEY=your_api_key_here
WANDB_PROJECT=aasist3_validation

# Hardware configuration
CUDA_VISIBLE_DEVICES=0

Writing .env


In [ ]:
# Check if file was sucessfuly created
!ls -a | grep .env
!cat .env

.env
# Path to the ASVspoof 2019/2021/2024 dataset
DATASET_DIR=/path/to/your/dataset

# Path to the pre-trained weights or the model you want to validate
CHECKPOINT_PATH=./models/weights.pth

# Optional: WandB configuration for logging validation metrics
WANDB_API_KEY=your_api_key_here
WANDB_PROJECT=aasist3_validation

# Hardware configuration
CUDA_VISIBLE_DEVICES=0


In [ ]:
from torch.utils.data import DataLoader

# 3. Setup DataLoader
# Ensure your Dataset class handles the ASVspoof protocol format
eval_dataset = ASVspoofDataset(protocol_path="path/to/eval_protocol.txt")
data_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# 4. Define Loss Function and Paths
loss_fn = nn.CrossEntropyLoss()
save_path = "output_scores.txt"
trial_path = "path/to/eval_trials.txt"

In [ ]:
from utils.validation import produce_evaluation_file

produce_evaluation_file(
    data_loader=data_loader,
    model=model,
    device=device,
    loss_fn=loss_fn,
    save_path=save_path,
    trial_path=trial_path,
    random=False,
    dropout=0,
    max_batches=None # Set an integer if you want a partial run for debugging
)

In [ ]:
from utils.validation import compute_scores
compute_scores()

---

## Alternative strategy 2: Importing metrics file

Download oficial evaluation metrics from AVSpoof2021 contest.

In [ ]:
!wget https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py

--2026-03-28 23:28:22--  https://raw.githubusercontent.com/asvspoof-challenge/2021/main/eval-package/eval_metrics.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18819 (18K) [text/plain]
Saving to: ‘eval_metrics.py’

eval_metrics.py     100%[===================>]  18.38K  --.-KB/s    in 0s      

2026-03-28 23:28:22 (187 MB/s) - ‘eval_metrics.py’ saved [18819/18819]



In [ ]:
# Load Protocol (cols: Speaker, Utterance, -, Attack, Label)
# Example file: ASVspoof2019.LA.cm.test.trl.txt
protocol_path = "/protocols/ASVspoof2019.LA.cm.test.trl.txt"
protocol = pd.read_csv(protocol_path, sep=" ", header=None,
                       names=['speaker', 'utt_id', 'dash', 'system', 'label'])

# Load your generated CM scores
cm_scores = pd.read_csv(cm_scores_file, sep=" ", header=None, names=['utt_id', 'score'])

# Merge to align Ground Truth with your Predictions
eval_data = pd.merge(protocol, cm_scores, on='utt_id')

# Split into Target (Bonafide) and Non-Target (Spoof) for EER calculation
target_scores = eval_data[eval_data['label'] == 'bonafide']['score'].values
nontarget_scores = eval_data[eval_data['label'] == 'spoof']['score'].values